# 🛡️ Module 11 — Exceptions & Error Handling

## 🎯 What you will learn
1. What exceptions are; the exception hierarchy
2. `try / except / else / finally` — every clause and their order
3. Catching multiple exceptions, `as e`, re-raising, exception chaining (`raise ... from`)
4. `raise` and `assert`
5. Custom exceptions
6. Best practices (EAFP vs LBYP, what NOT to do)
7. Debugging tools: `traceback`, `logging`, `pdb`
8. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐⭐⭐ — try/except/finally output prediction is guaranteed.

---
## 1. What is an exception?
When Python hits an error at run time it creates an **exception object** and stops normal flow. If nobody handles it, the program crashes with a traceback. Handling lets you recover gracefully.

In [1]:
def divide(a, b):
    return a / b

try:
    print(divide(10, 0))
except ZeroDivisionError:
    print("You can't divide by zero!")
print("program continues")

You can't divide by zero!
program continues


## 2. Common built-in exceptions

| Exception | Raised when… | Example |
|---|---|---|
| `SyntaxError` | invalid syntax (before run) | `print("hi"` |
| `NameError` | name not defined | `print(x)` |
| `TypeError` | wrong type of operand | `"a" + 1` |
| `ValueError` | right type, bad value | `int("abc")` |
| `ZeroDivisionError` | divide/modulo by 0 | `1 / 0` |
| `IndexError` | sequence index out of range | `[1][5]` |
| `KeyError` | dict key missing | `{}["a"]` |
| `AttributeError` | object has no such attribute | `(5).foo` |
| `FileNotFoundError` | file doesn't exist | `open("x")` |
| `ImportError`/`ModuleNotFoundError` | import fails | `import nope` |
| `IndentationError` | bad indentation | |
| `RecursionError` | recursion too deep | |
| `StopIteration` | iterator exhausted | `next(iter([]))` |
| `OverflowError` | number too large (floats) | `2.0 ** 10000` |
| `KeyboardInterrupt` | user pressed Ctrl+C | |
| `AssertionError` | `assert` failed | |
| `MemoryError`, `PermissionError`, `OSError`, `UnicodeDecodeError` … | | |

Let's trigger the popular ones deliberately and read their messages:

In [2]:
def trigger(code):
    try:
        exec(code, {})
    except Exception as e:
        print(f"{type(e).__name__:22} {e}")

for c in ['print(undefined_var)', '"a" + 1', 'int("abc")', '1/0', '[1,2][9]',
          '{"a":1}["b"]', '(5).foo', 'open("no_such_file.txt")', 'import nope',
          'next(iter([]))', 'float("x")', '"abc".index("z")', 'len(5)', 'int(None)',
          '[].pop()', '{1,2}[0]', 'a, b = [1,2,3]']:
    trigger(c)

NameError              name 'undefined_var' is not defined
TypeError              can only concatenate str (not "int") to str
ValueError             invalid literal for int() with base 10: 'abc'
ZeroDivisionError      division by zero
IndexError             list index out of range
KeyError               'b'
AttributeError         'int' object has no attribute 'foo'
FileNotFoundError      [Errno 2] No such file or directory: 'no_such_file.txt'
ModuleNotFoundError    No module named 'nope'
StopIteration          
ValueError             could not convert string to float: 'x'
ValueError             substring not found
TypeError              object of type 'int' has no len()
TypeError              int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
IndexError             pop from empty list
TypeError              'set' object is not subscriptable
ValueError             too many values to unpack (expected 2)


<string>:1: SyntaxWarning: 'set' object is not subscriptable; perhaps you missed a comma?


## 3. The hierarchy (exam favourite)
```
BaseException
 ├── SystemExit, KeyboardInterrupt, GeneratorExit
 └── Exception
      ├── ArithmeticError  ── ZeroDivisionError, OverflowError, FloatingPointError
      ├── LookupError      ── IndexError, KeyError
      ├── ValueError       ── UnicodeError ...
      ├── TypeError, NameError (UnboundLocalError), AttributeError
      ├── OSError          ── FileNotFoundError, PermissionError, TimeoutError ...
      ├── ImportError      ── ModuleNotFoundError
      ├── RuntimeError     ── RecursionError, NotImplementedError
      ├── StopIteration, AssertionError, EOFError, MemoryError ...
```
Catching a **parent** catches all its children. Catch `Exception`, never bare `except:` (it also traps Ctrl+C).

In [3]:
print(ZeroDivisionError.__mro__)
print(issubclass(KeyError, LookupError), issubclass(KeyboardInterrupt, Exception))

(<class 'ZeroDivisionError'>, <class 'ArithmeticError'>, <class 'Exception'>, <class 'BaseException'>, <class 'object'>)
True False


## 4. `try / except / else / finally`

```
try:
    risky code
except SomeError as e:
    runs if that error happened
else:
    runs ONLY if no exception happened
finally:
    ALWAYS runs (cleanup), even after return/break/uncaught error
```

In [4]:
def read_number(text):
    try:
        n = int(text)
    except ValueError:
        print("  -> invalid number")
        return None
    else:
        print("  -> conversion worked")
        return n
    finally:
        print("  -> finally always runs")

print(read_number("42"))
print(read_number("x"))

  -> conversion worked
  -> finally always runs
42
  -> invalid number
  -> finally always runs
None


### Multiple `except` blocks — order matters (specific before general)

In [5]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "zero!"
    except TypeError:
        return "bad types!"
    except (ValueError, OverflowError) as e:      # group in a tuple
        return f"value problem: {e}"
    except Exception as e:                         # catch-all safety net
        return f"unexpected: {e!r}"

print(safe_divide(4, 2), safe_divide(1, 0), safe_divide("a", 1))

2.0 zero! bad types!


### `finally` beats `return`

In [6]:
def f():
    try:
        return "try"
    finally:
        print("cleanup runs first")
print(f())

def g():
    try:
        return "try"
    finally:
        return "finally wins!"       # overrides (usually a bad idea)
print(g())

def h():
    for i in range(3):
        try:
            if i == 1:
                continue
            print("body", i)
        finally:
            print("finally", i)
h()

cleanup runs first
try
finally wins!
body 0
finally 0
finally 1
body 2
finally 2


## 5. Raising exceptions: `raise`

In [7]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError("age must be an integer")
    if not 0 <= age <= 150:
        raise ValueError(f"age {age} out of range 0-150")
    return age

for a in [25, -3, "x"]:
    try:
        print(set_age(a))
    except (TypeError, ValueError) as e:
        print(type(e).__name__, "->", e)

# Re-raise: log and pass on
def process():
    try:
        1 / 0
    except ZeroDivisionError:
        print("logging the problem...")
        raise                      # re-raises the same exception

try:
    process()
except ZeroDivisionError:
    print("caught after re-raise")

25
ValueError -> age -3 out of range 0-150
TypeError -> age must be an integer
logging the problem...
caught after re-raise


### Exception chaining: `raise NewError from original`

In [8]:
def load_config(text):
    try:
        return int(text)
    except ValueError as e:
        raise RuntimeError("Config is corrupt") from e

try:
    load_config("abc")
except RuntimeError as e:
    print(e, "| caused by:", repr(e.__cause__))

Config is corrupt | caused by: ValueError("invalid literal for int() with base 10: 'abc'")


## 6. `assert` — sanity checks for developers
`assert condition, message` raises `AssertionError` if condition is False. It's for **bugs**, not user input. Assertions are removed when Python runs with `-O`.

In [9]:
def average(nums):
    assert len(nums) > 0, "nums must not be empty"
    return sum(nums) / len(nums)

print(average([1, 2, 3]))
try:
    average([])
except AssertionError as e:
    print("AssertionError:", e)

2.0
AssertionError: nums must not be empty


## 7. Custom exceptions
Inherit from `Exception`. Give them meaningful names & extra data.

In [10]:
class InsufficientFundsError(Exception):
    def __init__(self, balance, amount):
        self.balance, self.amount = balance, amount
        super().__init__(f"Balance {balance} is less than requested {amount}")

class BankError(Exception):
    """Base class for all bank errors."""

class AccountLockedError(BankError): pass

class Account:
    def __init__(self, balance): self.balance = balance
    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount
        return self.balance

acct = Account(100)
try:
    acct.withdraw(500)
except InsufficientFundsError as e:
    print("Error:", e, "| short by", e.amount - e.balance)

try:
    raise AccountLockedError("locked")
except BankError as e:              # catches subclass via the base
    print("caught via base:", type(e).__name__)

Error: Balance 100 is less than requested 500 | short by 400
caught via base: AccountLockedError


## 8. Real-world patterns

In [11]:
# 1. Validate input in a loop
def ask_positive(values):
    for v in values:                 # simulated user inputs
        try:
            n = float(v)
            if n <= 0:
                raise ValueError("must be positive")
            return n
        except ValueError as e:
            print(f"  '{v}' rejected ({e})")
    return None
print(ask_positive(["abc", "-4", "7.5"]))

# 2. File handling with finally
import os
def read_first_line(path):
    f = None
    try:
        f = open(path)
        return f.readline()
    except FileNotFoundError:
        return "(file missing)"
    finally:
        if f: f.close()
print(read_first_line("does_not_exist.txt"))

# 3. EAFP vs LBYP
d = {"a": 1}
# LBYP (Look Before You Leap)
if "b" in d: v = d["b"]
else: v = None
# EAFP (Easier to Ask Forgiveness than Permission) - the Pythonic way
try: v = d["b"]
except KeyError: v = None
print(v)

# 4. retry with exponential backoff (concept)
import time
def unreliable(state={"n": 0}):
    state["n"] += 1
    if state["n"] < 3: raise ConnectionError("network down")
    return "connected"
for attempt in range(1, 5):
    try:
        print(unreliable(), "on attempt", attempt); break
    except ConnectionError as e:
        print("retry after error:", e); time.sleep(0.01 * 2 ** attempt)

  'abc' rejected (could not convert string to float: 'abc')
  '-4' rejected (must be positive)
7.5
(file missing)
None
retry after error: network down
retry after error: network down
connected on attempt 3


## 9. Debugging tools

In [12]:
import traceback, logging
try:
    {}["missing"]
except KeyError:
    tb = traceback.format_exc()
    print(tb.strip().splitlines()[-1])

logging.basicConfig(level=logging.ERROR, format="%(levelname)s %(message)s", force=True)
try:
    1 / 0
except ZeroDivisionError:
    logging.exception("math failed")      # logs message AND full traceback

# In a terminal: `import pdb; pdb.set_trace()` or simply `breakpoint()` pauses in the debugger.
# Commands: n (next) s (step into) c (continue) p var (print) l (list) q (quit)

KeyError: 'missing'


ERROR math failed
Traceback (most recent call last):
  File "<11_Exceptions#23>", line 10, in <module>
ZeroDivisionError: division by zero


## 10. Best practices ✅ / ❌

| ✅ Do | ❌ Don't |
|---|---|
| Catch **specific** exceptions | bare `except:` or `except Exception: pass` (swallows bugs) |
| Keep the `try` block **small** | wrap 50 lines in one try |
| Use `finally` / `with` for cleanup | forget to close files/connections |
| Add context when re-raising (`from e`) | lose the original traceback |
| Raise exceptions early with clear messages | return error codes like `-1` or `None` silently |
| Log errors | print and ignore |

---
# 🕵️ Hidden Facts & Pro Tricks

In [13]:
# 1. `contextlib.suppress` for "I expect this may fail, ignore it"
from contextlib import suppress
import os
with suppress(FileNotFoundError):
    os.remove("ghost.txt")

# 2. Exception groups & except* (Python 3.11+): handle several errors at once
try:
    raise ExceptionGroup("multiple", [ValueError("v"), TypeError("t")])
except* ValueError as eg:
    print("value errors:", [str(e) for e in eg.exceptions])
except* TypeError as eg:
    print("type errors:", [str(e) for e in eg.exceptions])

# 3. add_note (3.11+): attach extra info to an exception
try:
    try:
        int("x")
    except ValueError as e:
        e.add_note("while parsing user input on line 12")
        raise
except ValueError as e:
    print(e.__notes__)

# 4. Exceptions carry attributes
try:
    open("nofile.txt")
except OSError as e:
    print(e.errno, e.strerror, e.filename)

# 5. StopIteration is how for-loops end; NotImplementedError marks abstract methods
def todo(): raise NotImplementedError("subclass must implement")

# 6. `try` can be used as an expression-y idiom with next()/get() defaults to avoid exceptions
print(next(iter([]), "empty"), {}.get("x", "default"), getattr(object(), "nope", "none"))

# 7. A `while True` + try/except/else/break loop is the standard input-validation pattern
# 8. `sys.exit()` raises SystemExit; `except Exception` does NOT catch it (by design)
import sys
try:
    sys.exit(0)
except SystemExit as e:
    print("SystemExit caught explicitly, code =", e.code)

# 9. Warnings are not exceptions: warnings.warn("careful", DeprecationWarning)
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    warnings.warn("deprecated!", DeprecationWarning)
    print(len(w), w[0].category.__name__)

# 10. Python 3.14+ / 3.12+ give friendlier error messages with suggestions
try:
    "abc".uper()
except AttributeError as e:
    print(e)

value errors: ['v']
type errors: ['t']
['while parsing user input on line 12']
2 No such file or directory nofile.txt
empty default none
SystemExit caught explicitly, code = 0
1 DeprecationWarning
'str' object has no attribute 'uper'


**Hidden fact:** In Python, using exceptions for normal flow is *cheap* — `try` costs almost nothing when no error occurs (zero-cost exceptions since 3.11). That's why EAFP is idiomatic, unlike in Java.

**Hidden fact:** `except Exception as e:` deletes `e` at the end of the block (to avoid reference cycles). Copy it if you need it later: `err = e`.

---
# 📝 Exam Corner

**Theory**
1. What is an exception? Difference between error and exception; syntax vs runtime errors.
2. Explain `try-except-else-finally` with an example. When does `else` run?
3. How do you raise an exception? What is `assert`?
4. How do you create a user-defined exception?
5. What happens if an exception is not handled?
6. Difference between `raise` and `assert`.

**Output prediction**

In [14]:
def test(x):
    try:
        print("A", end=" ")
        r = 10 / x
        print("B", end=" ")
    except ZeroDivisionError:
        print("C", end=" ")
        return "err"
    else:
        print("D", end=" ")
        return r
    finally:
        print("E", end=" ")

print(test(2))
print(test(0))

try:
    try:
        raise ValueError("inner")
    finally:
        print("cleanup")
except ValueError as e:
    print("outer caught", e)

A B D E 5.0
A C E err
cleanup
outer caught inner


<details><summary>💡 Answers</summary>

```
A B D E 5.0
A C E err
cleanup
outer caught inner
```
</details>

---
## ✅ Summary
- `try` (risky) → `except` (handle) → `else` (no error) → `finally` (always).
- Specific exceptions first; never a bare `except:`.
- `raise` to signal problems, `raise X from e` to chain, `assert` for internal checks.
- Custom exceptions inherit `Exception`; group them under a base class.

➡️ **Next:** Module 12 — File Handling.